<a href="https://colab.research.google.com/github/chaseshallow/Ansci4040FinalClassProject/blob/main/Step04aCropTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 04a · Cut out the cow: test (4 minutes)

Takes the tracks from **Step 03a** (`Step03aTrackTest/Tracks.csv`) and cuts the cow out of the clip, at least 8 crops per kept track and at least one every 15 s (~16 crops, ~1 min):

1. drops duplicate boxes (a weaker box inside a stronger one in the same frame),
2. keeps only tracks that are long and confident enough,
3. picks frames spread over each track (preferring frames where YOLO agrees with SAM), so the crops show different angles,
4. masks each cow with SAM 2.1 (the pen turns grey; a calf touching her, e.g. being licked, can stay in) and saves the crop.

Writes into the Shared Drive, `CalvingPenPipelineOutputs/<video>/Step04aCropTest/`, **overwriting** each run:

- `KeptTracks.csv`: every track, kept or not and why; `RelativeSize` is box size vs. a typical track (information only)
- `Crops.csv` + `Crops/Track<id>/`: the masked crops: **input for step 05a**
- `Sheets/Overview.jpg` (one crop per kept track) and `Sheets/Track<id>.jpg` (all crops of a track): check these by eye

Needs a **T4 GPU** for SAM. The folder is shared: one person runs a step at a time. Edit cell 2 and rerun; don't add cells.

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub, install SAM.
from google.colab import drive
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/chaseshallow/Ansci4040FinalClassProject.git"
RepoBranch = "main"
RepoDir = "/content/Ansci4040FinalClassProject"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
%pip install -q -r "{RepoDir}/CalvingPen/requirements.txt"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline.Crop import CropCows
from CalvingPipeline.Paths import OutputRoot
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

In [ ]:
# 2. Settings: the only cell to edit.
VideoStem = "2026-09-26 00-00-00 Calving Pen 2"
VideoOutputDir = OutputRoot / VideoStem   # Shared Drive: CalvingPenPipelineOutputs/<video>/
ClipPath = VideoOutputDir / "ReferenceClip.mp4"   # step 01
TracksPath = VideoOutputDir / "Step03aTrackTest" / "Tracks.csv"   # step 03a; old tracker: "BaselineByteTrack" (then MinFrames = 25)
OutputDir = VideoOutputDir / "Step04aCropTest"

MinFrames = 10         # drop tracks shorter than this (10 frames = 5 s at 2 fps)
MinConfidence = 0.4    # drop tracks whose average confidence is below this
CropsPerTrack = 8     # at least this many crops per kept track, spread over its time span
CropEverySeconds = 15  # and at least one crop every 15 s (a SAM track can last the whole hour)
UseMasks = True        # False = plain box crops, no SAM (faster, background kept)

assert TracksPath.exists(), f"Run step 03a first: {TracksPath}"

In [ ]:
# 3. Run: crop + mask, then show which tracks were kept and the overview sheet.
import pandas as pd
from IPython.display import Image, display

Stats = CropCows(ClipPath, TracksPath, OutputDir, MinFrames=MinFrames, MinConfidence=MinConfidence,
                 CropsPerTrack=CropsPerTrack, CropEverySeconds=CropEverySeconds, UseMasks=UseMasks)

display(pd.read_csv(OutputDir / "KeptTracks.csv"))
display(Image(str(OutputDir / "Sheets" / "Overview.jpg")))
print("Per-track sheets:", OutputDir / "Sheets")

## About this step

**Model:** **SAM 2.1 Small in image mode** (Meta's Segment Anything 2.1), given each box as a hint, outlines the animal; everything outside the outline is painted grey.

**Rules:** duplicate boxes removed; tracks need ≥ `MinFrames` frames and average confidence ≥ `MinConfidence`; crops are spread over each track's time span (at least half a slice apart). `RelativeSize` is information only: a cow seen from behind is as small as a calf.

**What we found (earlier, on the ByteTrack tracks):** 167 duplicate boxes removed, 14 of 40 tracks kept, **306 crops, all the same cow**; SAM cut her out cleanly. When she licks the calf (~2560–3220 s), the calf ends up inside her mask. To be redone on the SAM tracks.

Full write-up, glossary and all models: `CalvingPen/README.md` in the repo.